# import cv2
import numpy as np

VIDEO_PATH = "dataset/road_video_proper.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("ERROR: road_video.mp4 not found")
else:

    departure_count = 0

    previous_offset = None
    previous_lane_center = None

    OFFSET_THRESHOLD = 0.10
    CONFIDENCE_THRESHOLD = 0.50
    REQUIRED_FRAMES = 5

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        height, width = frame.shape[:2]

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        blur = cv2.GaussianBlur(gray, (5, 5), 0)

        edges = cv2.Canny(blur, 50, 150)

        mask = np.zeros_like(edges)

        roi_points = np.array([[
            (0, height),
            (int(width * 0.30), int(height * 0.55)),
            (int(width * 0.70), int(height * 0.55)),
            (width, height)
        ]], dtype=np.int32)

        cv2.fillPoly(mask, roi_points, 255)

        roi = cv2.bitwise_and(edges, mask)

        lines = cv2.HoughLinesP(
            roi,
            1,
            np.pi / 180,
            30,
            minLineLength=60,
            maxLineGap=50
        )

        vehicle_center = width / 2

        reference_y = int(height * 0.85)

        left_lines = []
        right_lines = []

        if lines is not None:

            for line in lines:

                x1, y1, x2, y2 = np.array(line).flatten()

                if x2 == x1 or y2 == y1:
                    continue

                slope = (y2 - y1) / (x2 - x1)

                if abs(slope) < 0.4:
                    continue

                x_ref = x1 + (
                    (reference_y - y1) *
                    (x2 - x1) /
                    (y2 - y1)
                )

                if slope < 0 and x_ref < vehicle_center:
                    left_lines.append((x_ref, x1, y1, x2, y2))

                elif slope > 0 and x_ref > vehicle_center:
                    right_lines.append((x_ref, x1, y1, x2, y2))

        result = frame.copy()

        valid_lane = False

        if len(left_lines) > 0 and len(right_lines) > 0:

            left_lines.sort(key=lambda x: abs(vehicle_center - x[0]))
            right_lines.sort(key=lambda x: abs(vehicle_center - x[0]))

            lx_ref, lx1, ly1, lx2, ly2 = left_lines[0]
            rx_ref, rx1, ry1, rx2, ry2 = right_lines[0]

            left_m = (lx2 - lx1) / (ly2 - ly1)
            left_b = lx1 - left_m * ly1

            right_m = (rx2 - rx1) / (ry2 - ry1)
            right_b = rx1 - right_m * ry1

            left_x = left_m * reference_y + left_b
            right_x = right_m * reference_y + right_b

            lane_width = right_x - left_x

            lane_center = (left_x + right_x) / 2

            if width * 0.25 < lane_width < width * 0.75:

                valid_lane = True

                pixel_offset = vehicle_center - lane_center

                normalized_offset = abs(pixel_offset) / lane_width

                confidence = 1.0

                if lane_width < width * 0.30:
                    confidence -= 0.20

                if lane_width > width * 0.70:
                    confidence -= 0.20

                if previous_lane_center is not None:

                    center_change = abs(
                        lane_center - previous_lane_center
                    )

                    if center_change > width * 0.10:
                        confidence -= 0.30

                confidence = max(
                    0.0,
                    min(1.0, confidence)
                )

                if confidence >= CONFIDENCE_THRESHOLD:

                    if normalized_offset > OFFSET_THRESHOLD:

                        departure_count += 1

                    else:

                        departure_count = 0

                else:

                    departure_count = 0

                if confidence < CONFIDENCE_THRESHOLD:

                    warning = "LANE UNCERTAIN"

                elif departure_count >= REQUIRED_FRAMES:

                    warning = "LANE DEPARTURE WARNING"

                elif normalized_offset > OFFSET_THRESHOLD:

                    warning = "POSSIBLE DEPARTURE"

                else:

                    warning = "NORMAL"

                previous_offset = normalized_offset
                previous_lane_center = lane_center

                cv2.line(
                    result,
                    (int(left_x), reference_y),
                    (int(left_x), height),
                    (0, 255, 0),
                    5
                )

                cv2.line(
                    result,
                    (int(right_x), reference_y),
                    (int(right_x), height),
                    (0, 255, 0),
                    5
                )

                cv2.line(
                    result,
                    (int(lane_center), reference_y),
                    (int(lane_center), height),
                    (255, 0, 0),
                    4
                )

                cv2.line(
                    result,
                    (int(vehicle_center), reference_y),
                    (int(vehicle_center), height),
                    (0, 0, 255),
                    4
                )

                cv2.putText(
                    result,
                    f"Normalized Offset: {normalized_offset:.3f}",
                    (25, 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 0),
                    2
                )

                cv2.putText(
                    result,
                    f"Confidence: {confidence:.2f}",
                    (25, 65),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 0),
                    2
                )

                cv2.putText(
                    result,
                    f"Departure Frames: {departure_count}/{REQUIRED_FRAMES}",
                    (25, 95),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 0),
                    2
                )

                cv2.putText(
                    result,
                    warning,
                    (25, 140),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.85,
                    (0, 0, 255),
                    3
                )

            else:

                departure_count = 0
                previous_lane_center = None

                cv2.putText(
                    result,
                    "INVALID LANE WIDTH",
                    (25, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

        else:

            departure_count = 0
            previous_lane_center = None

            cv2.putText(
                result,
                "LANE NOT DETECTED",
                (25, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

        cv2.imshow(
            "Step 6 - Temporal Consistency + LDW",
            result
        )

        key = cv2.waitKey(25) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()